# Problems writeup

## Byte-Pair Encoding (BPE) Tokenizer

### The Unicode Standard

**Problem (**`unicode1`**): Understanding Unicode (1 point)**



- What Unicode character does `chr(0)` return?

  > It returns the character named `NUL`

- How does this character's string representation (`__repr__()`) differ from its printed representation?
  > The `__repr__` is the encoding of the printed representation 
  > with printable characters, especially with the backslash escaped
  > as well as the single quotes included.

In [2]:
print(
    f"chr(0)->[{chr(0)}]\n"
    f"chr(0).__repr__()->[{chr(0).__repr__()}]"
)
chr(0).__repr__()

chr(0)->[ ]
chr(0).__repr__()->['\x00']


"'\\x00'"

- What happens when this character occurs in text? 
  It may be helpful to play around with 
  the following in your Python interpreter 
  and see if it matches your expectations:

In [3]:
chr(0)

'\x00'

In [4]:
print(chr(0))

 


In [5]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [6]:
if not print("this is a test" + chr(0) + "string", sep="+"):
    from io import StringIO
    with StringIO() as s_:
        print("this is a test" + chr(0) + "string", file=s_)
        s = s_.getvalue()
s

this is a test string


'this is a test\x00string\n'

In [7]:
print(s)

this is a test string



The tests reveal that a string containing `NUL` 
get treated variously by different front ends; 
Jupyter displays it as a whitespace
while Python REPL simply omits it.
Neither treats it the C way, though Windows clipboard API does.

### Unicode Encodings

**Problem (**`unicode2`**): Unicode Encodings (3 points)**

- What are some reasons to prefer 
  training our tokenizer on UTF-8 encoded bytes, 
  rather than UTF-16 or UTF-32? 
  It may be helpful to compare the output of these encodings for various input strings.

  > UTF32 is constant length encoding but too wasteful for ASCII; UTF16 is legacy cringe.

- Consider the following (incorrect) function, 
  which is intended to decode a UTF-8 byte string into a Unicode string.
  Why is this function incorrect? 
  Provide an example of an input byte string that yields incorrect results.

In [8]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

Not every UT8 code unit (i.e. a 8-bit byte) is a encodes a Unicode code point.

See [[Accepted] What's the difference between a character, a code point, a glyph and a grapheme? - Stack Overflow](https://stackoverflow.com/questions/27331819/whats-the-difference-between-a-character-a-code-point-a-glyph-and-a-grapheme/27331885#27331885)

In [9]:
import sys, traceback  # noqa: E401
try:
    decode_utf8_bytes_to_str_wrong("こんにちは".encode())
except UnicodeDecodeError:
    traceback.print_exc(file=sys.stdout)

Traceback (most recent call last):
  File "/tmp/ipykernel_59400/1442419784.py", line 3, in <module>
    decode_utf8_bytes_to_str_wrong("こんにちは".encode())
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_59400/488356226.py", line 2, in decode_utf8_bytes_to_str_wrong
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
                    ~~~~~~~~~~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data


- Give a two-byte sequence that does not decode to any Unicode character(s).

In [10]:
for b in [
        b"\x80\x80",     # lone continuation
        b"\xE2\x82",     # missing continuation
        b"\xC1\xBF",     # forbidden lead - overlong encoding of U+007F
        b"\xF0\x80",     # overlong encoding of U+0000
        b"\xF5\x80",     # out of range
    ]:
    try:
        print(b.decode("utf-8"))
    except UnicodeDecodeError as e:
        print("\n", e.object, e.__traceback__, flush=True)
        traceback.print_exc(file=sys.stdout)


 b'\x80\x80' <traceback object at 0x7854583ee2c0>
Traceback (most recent call last):
  File "/tmp/ipykernel_59400/3489787588.py", line 9, in <module>
    print(b.decode("utf-8"))
          ~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

 b'\xe2\x82' <traceback object at 0x7854583ee2c0>
Traceback (most recent call last):
  File "/tmp/ipykernel_59400/3489787588.py", line 9, in <module>
    print(b.decode("utf-8"))
          ~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 0-1: unexpected end of data

 b'\xc1\xbf' <traceback object at 0x7854583ee2c0>
Traceback (most recent call last):
  File "/tmp/ipykernel_59400/3489787588.py", line 9, in <module>
    print(b.decode("utf-8"))
          ~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc1 in position 0: invalid start byte

 b'\xf0\x80' <traceback object at 0x78545828c580>
Traceback (most recent call last):
  File "/tmp/ip

### Sub-word Tokenization

Sub-word tokenization is a midpoint between word-level tokenizers and byte-level tokenizers.

In this assignment, we'll implement a byte-level BPE tokenizer.  
The process of constructing the BPE tokenizer vocabulary 
is known as "training" the BPE tokenizer.

### BPE Tokenizer Training

- Vocabulary initialization
- Pre-tokenization
  - The original BPE implementation pre-tokenizes by simply splitting on whitespace
  - Most modern tokenizers use a regex-based pre-tokenizer, like

In [6]:
# import base64
import requests

# retrieved from <https://github.com/openai/tiktoken/pull/234/changes>
url: str = "https://github.com/l0rinc/tiktoken/blob/6f261deef63b49a7da9000b57a7cf938d7315ab3/tiktoken_ext/openai_public.py#L23"
parts= url.split("github.com/", 1)[1].split("/")
owner: str = parts[0]
repo: str = parts[1]
ref_id: str = parts[3]
path: str = "/".join(parts[4:]).split("#", 1)[0]
# extract line number or line number range
# ln_no: int = int(parts[-1].split("#L", 1)[1])
lns= parts[-1].split("#L", 1)[1].split("-", 1)
ln_start: int = int(lns[0])
ln_end: int = int(lns[1]) if len(lns) > 1 else int(lns[0])

api: str = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
raw: str = f"https://raw.githubusercontent.com/{owner}/{repo}/{ref_id}/{path}"
params: dict[str, str] = {"ref": ref_id}

# data = requests.get(api, params=params).json() # rate limited, will fail for public proxy
# content_api = base64.b64decode(data["content"]).decode("utf-8")
# line_23_api = content_api.splitlines()[22]

text = requests.get(raw).text
lines = text.splitlines()[ln_start - 1:ln_end]
[print(line) for line in lines]
# text.splitlines()[ln_start - 1:ln_end]

# split json key and value by the first colon, and strip whitespace
kv: dict[str, str] = dict(line.split(":", 1) for line in lines)
kv = dict((k.strip(), v.strip()) for k, v in kv.items())

# oh no, we need to create a regex literal from a string literal
# giving up on this nightmare of security folks
PAT: str = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

        "pat_str": r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""",


In [10]:
# requires `regex` package
import regex as re
re.findall(PAT, "some text that i'll pre-tokenize")
[match.group() for match in
    re.finditer(PAT, "some text that i'll pre-tokenize")]

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

- Compute BPE merges

### Experimenting with BPE Tokenizer Training

In [ ]:
%%bash
set -euo pipefail

mkdir -p data/ && cd data/
mkdir -p bak/  && mv -b * bak/ 2>/dev/null || true

touch .gitignore && echo "*" > .gitignore

wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
gunzip owt_train.txt.gz
wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
gunzip owt_valid.txt.gz

cd ..

### Experimenting with BPE Tokenizer Training

#### 
**Problem (**`train_bpe`**): BPE Tokenizer Training (15 points)**

In [9]:
%uv --directory .. run pytest -v tests/test_train_bpe.py

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.2, pluggy-1.6.0 -- /workspaces/stanford-cs336.assignment1-basics/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /workspaces/stanford-cs336.assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 3 items                                                              

tests/test_train_bpe.py::test_train_bpe_speed PASSED
tests/test_train_bpe.py::test_train_bpe PASSED
tests/test_train_bpe.py::test_train_bpe_special_tokens PASSED

============================== 3 passed in 1.00s ===============================
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# prototyping the multiprocessing parameters
for cpu in [None, 0, 1, 2, 4, 20]:
    ns = [cpu or x or 1 for x in [-1, None, False, 0, True, 1, 2, 1000]]
    print([max(1, min(x or 1, n or 1)) for x, n in zip([-1, None, False, 0, True, 1, 2, 1000], ns)])

[1, 1, 1, 1, 1, 1, 2, 1000]
[1, 1, 1, 1, 1, 1, 2, 1000]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 1, 1, 1, 1, 1, 2, 2]
[1, 1, 1, 1, 1, 1, 2, 4]
[1, 1, 1, 1, 1, 1, 2, 20]


In [ ]:
# experimenting with str / bytes interactions
x = "hello world"
tuple(ch.encode() for ch in x)
b"Hello world"[2:4]
list(zip(x, x[1:]))
list(x.encode())

[104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100]

In [ ]:
# experimenting with zip()
x, y = zip(*{1: 'a', 2: 'b'}.items())
print(x, y)

(1, 2) ('a', 'b')


In [55]:
# test train_bpe on a simple string
import os
from importlib import reload
import cs336_basics.tokenizer
reload(cs336_basics.tokenizer)
train_bpe = cs336_basics.tokenizer.train_bpe
# create a small corpus file
with open("temp.txt", "w") as f:
    f.write("""low low low low low
lower lower widest widest widest
newest newest newest newest newest newest<|endoftext|>""")

f = "temp.txt"
f = "../tests/fixtures/tinystories_sample.txt"
vocab, merges = train_bpe(f, 256+20, ["<|endoftext|>"])
print([(id, vocab[id]) for id in range(256, len(vocab))])
print(merges)

# delete the temporary file
os.remove("temp.txt")

[(256, b'\xc3\xbf'), (257, b' t'), (258, b'he'), (259, b' a'), (260, b' s'), (261, b' w'), (262, b'nd'), (263, b'ed'), (264, b' f'), (265, b' the'), (266, b'it'), (267, b' to'), (268, b' d'), (269, b'is'), (270, b' b'), (271, b'll'), (272, b' wa'), (273, b' and'), (274, b'ou'), (275, b'ay')]
[(b' ', b't'), (b'h', b'e'), (b' ', b'a'), (b' ', b's'), (b' ', b'w'), (b'n', b'd'), (b'e', b'd'), (b' ', b'f'), (b' t', b'he'), (b'i', b't'), (b' t', b'o'), (b' ', b'd'), (b'i', b's'), (b' ', b'b'), (b'l', b'l'), (b' w', b'a'), (b' a', b'nd'), (b'o', b'u'), (b'a', b'y')]


In [11]:
# test train_bpe on a 5M sample from TinyStories
import os
from importlib import reload
import cs336_basics.tokenizer
reload(cs336_basics.tokenizer)
train_bpe = cs336_basics.tokenizer.train_bpe

f = "../tests/fixtures/tinystories_sample_5M.txt"
vocab, merges = train_bpe(f, 256+20, ["<|endoftext|>"])
print([(id, vocab[id]) for id in range(256, len(vocab))])
print(merges)

[(256, b'\xff'), (257, b'he'), (258, b' t'), (259, b' a'), (260, b' s'), (261, b' w'), (262, b'nd'), (263, b' the'), (264, b'ed'), (265, b' b'), (266, b' to'), (267, b' and'), (268, b' h'), (269, b' f'), (270, b'in'), (271, b' wa'), (272, b' T'), (273, b'it'), (274, b're'), (275, b'ou')]
[(b'h', b'e'), (b' ', b't'), (b' ', b'a'), (b' ', b's'), (b' ', b'w'), (b'n', b'd'), (b' t', b'he'), (b'e', b'd'), (b' ', b'b'), (b' t', b'o'), (b' a', b'nd'), (b' ', b'h'), (b' ', b'f'), (b'i', b'n'), (b' w', b'a'), (b' ', b'T'), (b'i', b't'), (b'r', b'e'), (b'o', b'u')]


####
**Problem (**`train_bpe_tinystories`**):  BPE Training on TinyStories (2 points)**

In [ ]:
from importlib import reload
import cs336_basics.tokenizer
reload(cs336_basics.tokenizer)
train_bpe = cs336_basics.tokenizer.train_bpe

f = "data/TinyStoriesV2-GPT4-train.txt"
vocab, merges = train_bpe(f, 10000, ["<|endoftext|>"])

KeyboardInterrupt: 

####
**Problem (**`train_bpe_expts_owt`**):  BPE Training on OpenWebText (2 points)**